In [ ]:
!pip install langchain tiktoken python-dotenv langchain-community sentence-transformers langchain-huggingface

In [ ]:
!pip install sentence-transformers

In [ ]:
pip install -qU langchain-huggingface

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings  = HuggingFaceEmbeddings(
  model_name = "ibm-granite/granite-embedding-97m-multilingual-r2"
)

d:\LLMOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 74/74 [00:00<00:00, 1542.28it/s]


### DATA INGESTION

In [6]:
from langchain_community.document_loaders import TextLoader

C:\Users\shubh\AppData\Local\Temp\ipykernel_7392\2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [7]:
loader = TextLoader("D:\\LLMOps\\data\\AI.txt", encoding="utf-8")
documents = loader.load()

In [8]:
documents

[Document(metadata={'source': 'D:\\LLMOps\\data\\AI.txt'}, page_content='Transformers and Large Language Models for Efficient Intrusion Detection\nSystems: A Comprehensive Survey\nHamza Kheddara,∗\naLSEA Laboratory, Department of Electrical Engineering, University of Medea, 26000, Algeria\nA R T I C L E I N F O\nKeywords:\nAnomalies detection\nCyber-security\nIntrusion detection\nLarge language model\nNatural language processing\nTransformers\nA B S T R A C T\nWith significant advancements in Transformers and large language models (LLMs), natural language\nprocessing (NLP) has extended its reach into many research fields due to its enhanced capabilities in text\ngeneration and user interaction. One field benefiting greatly from these advancements is cybersecurity. In\ncybersecurity, many parameters that need to be protected and exchanged between senders and receivers\nare in the form of text and tabular data, making NLP a valuable tool in enhancing the security measures\nof communicati

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=60)

In [11]:
chunks = splitter.split_documents(documents=documents)

In [12]:
chunks

[Document(metadata={'source': 'D:\\LLMOps\\data\\AI.txt'}, page_content='Transformers and Large Language Models for Efficient Intrusion Detection\nSystems: A Comprehensive Survey\nHamza Kheddara,∗\naLSEA Laboratory, Department of Electrical Engineering, University of Medea, 26000, Algeria\nA R T I C L E I N F O\nKeywords:\nAnomalies detection\nCyber-security\nIntrusion detection\nLarge language model\nNatural language processing\nTransformers\nA B S T R A C T\nWith significant advancements in Transformers and large language models (LLMs), natural language'),
 Document(metadata={'source': 'D:\\LLMOps\\data\\AI.txt'}, page_content='processing (NLP) has extended its reach into many research fields due to its enhanced capabilities in text\ngeneration and user interaction. One field benefiting greatly from these advancements is cybersecurity. In\ncybersecurity, many parameters that need to be protected and exchanged between senders and receivers\nare in the form of text and tabular data, ma

In [ ]:
pip install langchain-chroma chromadb

In [21]:
from langchain_chroma import Chroma

In [22]:
vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings)

In [23]:
docs = vector_store.similarity_search("What is Transformer?", k=4)

for i, doc in enumerate(docs):
  print(f"Document {i+1}:")
  print(doc.page_content)
  print("\n")


Document 1:
Transformer
CNN
Transformer
Transformer
Transformer
Figure 6: Various configurations of CNN-Transformer mechanisms include: (a) CNN and Transformer alternated; (b) One-by-one CNNTransformer feature fusion; (c) CNN combined with multi-scale Transformer feature fusion; (d) A sequence of CNN followed by a single
Transformer; (e) A sequence of CNN followed by the last CNN and a single Transformer feature fusion; (f) Sequential integration of


Document 2:
operations. They are renowned for their ability to detect features and patterns at various levels of abstraction, making them
pivotal in computer vision (CV) tasks. On the other hand, Transformers leverage self-Attention mechanisms to model long-range
dependencies, excelling in handling sequential data such as text and time series. They have revolutionized NLP and are increasingly


Document 3:
applied across various domains. By integrating CNNs for effective feature extraction and spatial hierarchy with Transformers’
capabili

In [24]:
retriever = vector_store.as_retriever()

In [25]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
  template="""
  You are a helpful AI assistant.

Use only the information provided in the Context section to answer the Question.

Instructions:
- If the answer is present in the Context, provide a clear and concise answer.
- If the Context does not contain enough information, say "I don't have enough information in the provided context to answer that."
- Do not make up facts or use outside knowledge.
- Cite relevant parts of the Context when appropriate.

Context:
{context}

Question:
{question}

Answer:
  """,
  input_variables = ["context", "question"]
)

In [27]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [ ]:
pip install langchain-groq

In [30]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="groq/compound-mini")

In [33]:
from langchain_core.runnables import RunnablePassthrough
chain = (
  {"context": retriever , "question": RunnablePassthrough()} | 
  prompt | llm | parser
)

In [35]:
chain.invoke("What is Transformer?")

'Transformer is a deep‑learning model that processes data using a self‑attention mechanism. By weighting the importance of different parts of the input, it can model long‑range dependencies and understand context and relationships within sequential data such as text and time‑series, making it especially effective for NLP tasks【2†L1-L4】【4†L1-L4】.'